In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import subprocess
import sys

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/shoe_project")
bundles = sorted(PROJECT.glob("shoe_absa_*.zip"))
if not bundles:
    raise FileNotFoundError(f"{PROJECT} 안에 shoe_absa_*.zip 파일이 없습니다.")

bundle = bundles[-1]
ROOT = Path("/content") / bundle.stem
ROOT.mkdir(exist_ok=True)

print("사용할 ZIP:", bundle.name)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(ROOT)

print((ROOT / "requirements.txt").read_text())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", str(ROOT / "requirements.txt")],
    check=True,
)

Mounted at /content/drive
사용할 ZIP: shoe_absa_20261123_094430.zip
sentence-transformers==6.3.0
transformers==5.23.0
tokenizers==0.23.3
nltk==3.9.1


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/shoe_absa_20261123_094430/requirements.txt'], returncode=0)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import nltk

from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 유형에서 GPU를 선택하세요.")

device = torch.device("cuda")
print("사용 GPU:", torch.cuda.get_device_name(0))

settings = json.loads((ROOT / "settings.json").read_text(encoding="utf-8"))

def read_frame(name):
    # "None"으로 이름 붙인 군집과 리뷰 원문의 문자열을 그대로 읽는다.
    return pd.read_csv(ROOT / f"{name}.csv.gz", keep_default_na=False)

shoe_train_df = read_frame("train")
shoe_test_df = read_frame("test")
check_reviews = read_frame("check_reviews")
check_df = read_frame("check_df")
check_passed = read_frame("check_passed")

with np.load(ROOT / "clusters.npz") as arrays:
    cluster_ids = arrays["cluster_ids"]
    cluster_centroids = arrays["cluster_centroids"]

cluster_to_aspect = {int(k): v for k, v in settings["cluster_to_aspect"].items()}
COSINE_THRESHOLD = settings["threshold"]
ASPECT_EN = settings["aspect_en"]
SENTIMENTS = settings["sentiments"]

assert len(shoe_train_df) == settings["train_count"]
assert len(shoe_test_df) == settings["test_count"]
assert len(check_passed) == settings["check_passed_count"]
assert set(cluster_ids) == set(cluster_to_aspect)
assert cluster_centroids.shape == (len(cluster_ids), 768)

sbert = SentenceTransformer(str(ROOT / "sbert"), device="cuda")
tokenizer = AutoTokenizer.from_pretrained(ROOT / "absa", local_files_only=True)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    ROOT / "absa", local_files_only=True
).to(device).eval()

id_to_label = {
    int(k): v.lower() for k, v in sentiment_model.config.id2label.items()
}
assert set(id_to_label.values()) == set(SENTIMENTS)

for resource in ["punkt", "punkt_tab"]:
    nltk.download(resource, quiet=True, raise_on_error=True)

# 같은 ZIP으로 다시 접속하면 같은 결과 폴더를 사용한다.
OUT = PROJECT / f"results_{settings['bundle_id']}"
OUT.mkdir(exist_ok=True)

print("Train / Test:", len(shoe_train_df), "/", len(shoe_test_df))
print("대표벡터:", cluster_centroids.shape, "| 임계값:", COSINE_THRESHOLD)
print("이전 감성 판정 결과:", len(check_passed), "문장")
print("결과 저장 위치:", OUT)

사용 GPU: NVIDIA A100-SXM4-80GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Train / Test: 76320 / 19080
대표벡터: (30, 768) | 임계값: 0.6
이전 감성 판정 결과: 82 문장
결과 저장 위치: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430


In [ ]:
@torch.inference_mode()
def predict_sentiments(frame, batch_size=32):
    labels = []

    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        targets = [ASPECT_EN[a] for a in batch["aspect_candidate"]]

        inputs = tokenizer(
            batch["sentence"].tolist(),
            text_pair=targets,
            padding=True,
            truncation="only_first",
            max_length=256,
            return_tensors="pt",
        ).to(device)

        ids = sentiment_model(**inputs).logits.argmax(dim=1).cpu().tolist()
        labels.extend(id_to_label[i] for i in ids)

    return labels


sentence_columns = [
    "review_id", "sentence_no", "sentence", "nearest_cluster",
    "cosine_similarity", "aspect_candidate", "sentiment",
]

def process_reviews(reviews):
    rows = []
    for review in reviews.itertuples(index=False):
        for no, sentence in enumerate(
            sent_tokenize(review.text, language="english"), start=1
        ):
            sentence = sentence.strip()
            if 3 <= len(sentence.split()) <= 60:
                rows.append((review.review_id, no, sentence))

    if not rows:
        return pd.DataFrame(columns=sentence_columns)

    frame = pd.DataFrame(rows, columns=["review_id", "sentence_no", "sentence"])
    vectors = sbert.encode(
        frame["sentence"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    similarities = vectors @ cluster_centroids.T
    nearest = similarities.argmax(axis=1)

    frame["nearest_cluster"] = cluster_ids[nearest]
    frame["cosine_similarity"] = similarities[np.arange(len(frame)), nearest]
    frame["aspect_candidate"] = frame["nearest_cluster"].map(cluster_to_aspect)

    keep = (
        frame["cosine_similarity"].ge(COSINE_THRESHOLD)
        & frame["aspect_candidate"].notna()
        & frame["aspect_candidate"].ne("None")
    )
    passed = frame.loc[keep].copy()
    passed["sentiment"] = predict_sentiments(passed)
    return passed[sentence_columns]


gpu_labels = predict_sentiments(check_passed)
comparison = check_passed[["sentence", "aspect_candidate", "sentiment"]].copy()
comparison["colab_sentiment"] = gpu_labels
matches = comparison["sentiment"].eq(comparison["colab_sentiment"])

print(f"로컬과 Colab의 감성 판정 일치: {matches.sum()} / {len(matches)}")
if not matches.all():
    display(comparison.loc[~matches])

로컬과 Colab의 감성 판정 일치: 82 / 82


In [ ]:
from tqdm.auto import tqdm

REVIEWS_PER_CHUNK = 500

def run_split(reviews, split):
    folder = OUT / f"{split}_chunks_{REVIEWS_PER_CHUNK}"
    folder.mkdir(exist_ok=True)
    paths = []

    starts = range(0, len(reviews), REVIEWS_PER_CHUNK)
    for start in tqdm(starts, desc=split):
        stop = min(start + REVIEWS_PER_CHUNK, len(reviews))
        path = folder / f"{start:06d}_{stop:06d}.parquet"
        paths.append(path)

        if path.exists():
            continue

        result = process_reviews(reviews.iloc[start:stop])
        temporary = path.with_suffix(".partial")
        result.to_parquet(temporary, index=False, engine="pyarrow")
        temporary.replace(path)

    return pd.concat(
        [pd.read_parquet(path) for path in paths],
        ignore_index=True,
    )

train_sentence_df = run_split(shoe_train_df, "train")
test_sentence_df = run_split(shoe_test_df, "test")

print("Train에서 채택한 문장:", len(train_sentence_df))
print("Test에서 채택한 문장:", len(test_sentence_df))
display(train_sentence_df.head())

train:   0%|          | 0/153 [00:00<?, ?it/s]

test:   0%|          | 0/39 [00:00<?, ?it/s]

Train에서 채택한 문장: 60513
Test에서 채택한 문장: 14803


,review_id,sentence_no,sentence,nearest_cluster,cosine_similarity,aspect_candidate,sentiment
0,0,3,His feet are a little narrow and these are pre...,18,0.610517,사이즈/핏,negative
1,2,2,My 5 year old said they are comfy and she love...,7,0.697625,착화감,positive
2,3,2,"They fit nice, a little difficult for her to f...",23,0.712251,사이즈/핏,positive
3,4,3,I wear them working landscaping every day and ...,7,0.629408,착화감,positive
4,4,4,They are super durable and super comfortable!,7,0.717494,착화감,positive


In [ ]:
feature_columns = [
    f"{aspect}_{sentiment}"
    for aspect in ASPECT_EN
    for sentiment in SENTIMENTS
]

def make_review_vectors(reviews, sentence_results):
    review_ids = pd.Index(reviews["review_id"], name="review_id")
    assert review_ids.is_unique

    if sentence_results.empty:
        X = pd.DataFrame(0, index=review_ids, columns=feature_columns)
    else:
        feature = (
            sentence_results["aspect_candidate"] + "_" + sentence_results["sentiment"]
        )
        counts = pd.crosstab(sentence_results["review_id"], feature)
        X = counts.reindex(
            index=review_ids, columns=feature_columns, fill_value=0
        ).gt(0)

    X = X.astype("uint8")
    X.columns.name = None
    y = reviews.set_index("review_id")["rating"].reindex(X.index)
    assert y.between(1, 5).all()
    return X, y

X_train, y_train = make_review_vectors(shoe_train_df, train_sentence_df)
X_test, y_test = make_review_vectors(shoe_test_df, test_sentence_df)

for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    assert X.index.equals(y.index)
    assert X.shape[1] == 27
    X.join(y).to_parquet(OUT / f"{name}_review_vectors.parquet")
    covered = X.any(axis=1)
    print(f"{name}: {X.shape}, aspect 추출 리뷰 {covered.sum():,}개 ({covered.mean():.1%})")

train: (76320, 27), aspect 추출 리뷰 39,554개 (51.8%)
test: (19080, 27), aspect 추출 리뷰 9,716개 (50.9%)


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 새 세션에서도 셀 1~2로 OUT 경로를 복원한 후 여기부터 실행할 수 있다.
train_vectors = pd.read_parquet(OUT / "train_review_vectors.parquet")
test_vectors = pd.read_parquet(OUT / "test_review_vectors.parquet")

X_train = train_vectors.drop(columns="rating")
y_train = train_vectors["rating"]
X_test = test_vectors[X_train.columns]
y_test = test_vectors["rating"]

models = {
    "Train median": DummyRegressor(strategy="median"),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
}

has_pos = X_test.filter(regex="_positive$").any(axis=1)
has_neg = X_test.filter(regex="_negative$").any(axis=1)

predictions = pd.DataFrame({"actual_rating": y_test})
predictions["group"] = np.select(
    [has_pos & has_neg, has_pos | has_neg],
    ["mixed", "single"],
    default="neutral_or_unassigned",
)

metrics = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predicted = np.clip(model.predict(X_test), 1, 5)
    predictions[model_name] = predicted
    absolute_error = predictions["actual_rating"].sub(predicted).abs()

    for group in ["all", "mixed", "single", "neutral_or_unassigned"]:
        errors = (
            absolute_error if group == "all"
            else absolute_error[predictions["group"].eq(group)]
        )
        metrics.append({
            "model": model_name,
            "group": group,
            "n_reviews": len(errors),
            "MAE": errors.mean(),
        })

metrics = pd.DataFrame(metrics)
rating_distribution = pd.crosstab(
    predictions["group"], predictions["actual_rating"]
).reindex(columns=[1, 2, 3, 4, 5], fill_value=0)

display(metrics.pivot(index="model", columns="group", values="MAE").round(3))
display(rating_distribution)

predictions.to_parquet(OUT / "test_predictions.parquet")
metrics.to_csv(OUT / "experiment1_mae.csv", index=False)
rating_distribution.to_csv(OUT / "test_group_rating_counts.csv")
print("실험 1 결과 폴더:", OUT)

group,all,mixed,neutral_or_unassigned,single
model,,,,
Linear regression,0.958,1.075,1.096,0.780
Random forest,0.938,1.049,1.079,0.757
Train median,1.024,1.676,0.885,1.102


actual_rating,1,2,3,4,5
group,,,,,
mixed,164,150,235,257,288
neutral_or_unassigned,1024,520,811,1329,6040
single,1048,651,838,1284,4441


실험 1 결과 폴더: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430


In [ ]:
error = (predictions["actual_rating"] - predictions["Random forest"]).abs()
print(f"1점 이내: {(error <= 1).mean():.1%}")
print(f"1점 초과: {(error > 1).mean():.1%}")

1점 이내: 75.7%
1점 초과: 24.3%
